# GTSF Quant Mentorship TSF Project

## Part 1: Data Collection and Cleaning

### Data Collection

In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
tickers = ["AAPL", "MSFT", "GOOGL", "NVDA"]
tsf_data = yf.download(tickers, start = "2007-01-01", end = "2025-12-31")
tsf_data.head()

[*********************100%***********************]  4 of 4 completed


Price          Close                                      High             \
Ticker          AAPL      GOOGL       MSFT      NVDA      AAPL      GOOGL   
Date                                                                        
2007-01-03  2.510900  11.605533  21.073975  0.551293  2.594197  11.830648   
2007-01-04  2.566632  11.994459  21.038679  0.548696  2.575321  12.011585   
2007-01-05  2.548354  12.092003  20.918701  0.514316  2.582811  12.099697   
2007-01-08  2.560939  12.002400  21.123367  0.518137  2.592700  12.158518   
2007-01-09  2.773675  12.050056  21.144541  0.508052  2.785960  12.118309   

Price                                 Low                                  \
Ticker           MSFT      NVDA      AAPL      GOOGL       MSFT      NVDA   
Date                                                                        
2007-01-03  21.349220  0.573296  2.453970  11.444700  20.749325  0.531582   
2007-01-04  21.151600  0.551293  2.511499  11.624395  20.777549  0.535249   
2007-01-05  20.996335  0.537847  2.528879  11.866638  20.784608  0.510649   
2007-01-08  21.243346  0.528068  2.555246  11.968149  20.841064  0.507287   
2007-01-09  21.299809  0.522414  2.551350  11.943330  20.982217  0.507441   

Price           Open                                      Volume             \
Ticker          AAPL      GOOGL       MSFT      NVDA        AAPL      GOOGL   
Date                                                                          
2007-01-03  2.585508  11.566070  21.109262  0.566420  1238319600  307951740   
2007-01-04  2.518391  11.640528  20.961047  0.549307   847260400  315188496   
2007-01-05  2.569927  11.975598  20.911643  0.535708   834741600  274609116   
2007-01-08  2.575621  12.104410  20.925754  0.516150   797106800  189985824   
2007-01-09  2.590302  12.048815  21.172772  0.518900  3349298400  215040744   

Price                             
Ticker          MSFT        NVDA  
Date                              
2007-01-03  76935100  1154820000  
2007-01-04  45774500   797298000  
2007-01-05  44607200  1243344000  
2007-01-08  50220200   657270000  
2007-01-09  44636600   764166000

### Data Cleaning

In [3]:
tsf_data.isna().sum()

Price   Ticker
Close   AAPL      0
        GOOGL     0
        MSFT      0
        NVDA      0
High    AAPL      0
        GOOGL     0
        MSFT      0
        NVDA      0
Low     AAPL      0
        GOOGL     0
        MSFT      0
        NVDA      0
Open    AAPL      0
        GOOGL     0
        MSFT      0
        NVDA      0
Volume  AAPL      0
        GOOGL     0
        MSFT      0
        NVDA      0
dtype: int64

***from here, we see that this dataset has no null values to handle***

In [4]:
tsf_data.dtypes

Price   Ticker
Close   AAPL      float64
        GOOGL     float64
        MSFT      float64
        NVDA      float64
High    AAPL      float64
        GOOGL     float64
        MSFT      float64
        NVDA      float64
Low     AAPL      float64
        GOOGL     float64
        MSFT      float64
        NVDA      float64
Open    AAPL      float64
        GOOGL     float64
        MSFT      float64
        NVDA      float64
Volume  AAPL        int64
        GOOGL       int64
        MSFT        int64
        NVDA        int64
dtype: object

***we also can observe that our dataset is consistent and all columns already have numeric datatypes***

In [5]:
tsf_data.index.duplicated().sum()

np.int64(0)

***we have no duplicated dates within the dataframe***

In [6]:
date_gaps = tsf_data.index.to_series().diff().dt.days

In [7]:
date_gaps = tsf_data.index.to_series().diff().dt.days
date_gaps[date_gaps > 5]  # anything over a weekend is suspicious


Series([], Name: Date, dtype: float64)

In [8]:
print(f"Start: {tsf_data.index.min()}, End: {tsf_data.index.max()}")
print(f"Total trading days: {len(tsf_data)}")

# 4. Check for zero volume days (can indicate data issues)
print((tsf_data['Volume'] == 0).sum())

Start: 2007-01-03 00:00:00, End: 2025-12-30 00:00:00
Total trading days: 4779
Ticker
AAPL     0
GOOGL    0
MSFT     0
NVDA     0
dtype: int64


In [9]:
# 5. Check for outliers in returns
returns = tsf_data['Close'].pct_change()
outliers = returns.stack()
print(outliers[outliers.abs() > 0.10])

Date        Ticker
2007-11-13  AAPL      0.105359
2008-01-07  NVDA     -0.103333
2008-01-16  NVDA     -0.112981
2008-01-23  AAPL     -0.106463
2008-02-14  NVDA     -0.163212
                        ...   
2024-07-31  NVDA      0.128121
2025-01-27  NVDA     -0.169682
2025-04-09  AAPL      0.153288
            MSFT      0.101337
            NVDA      0.187227
Length: 76, dtype: float64


***this series marks some important days where the change of the close price changed by 15% or more from 2007-2025***

In [10]:
# lets use these outliers to flag the data
log_returns = np.log(tsf_data['Close']/tsf_data['Close'].shift(1))

outlier_flags = log_returns.abs() > 0.10
outlier_flags.columns = pd.MultiIndex.from_tuples([('Outlier', col) for col in outlier_flags.columns])

In [11]:
#lets combine the dataframes now to actually show the flags
tsf_data = pd.concat([tsf_data, outlier_flags], axis =1)

In [12]:
tsf_data.head()

Close                                      High             \
                AAPL      GOOGL       MSFT      NVDA      AAPL      GOOGL   
Date                                                                        
2007-01-03  2.510900  11.605533  21.073975  0.551293  2.594197  11.830648   
2007-01-04  2.566632  11.994459  21.038679  0.548696  2.575321  12.011585   
2007-01-05  2.548354  12.092003  20.918701  0.514316  2.582811  12.099697   
2007-01-08  2.560939  12.002400  21.123367  0.518137  2.592700  12.158518   
2007-01-09  2.773675  12.050056  21.144541  0.508052  2.785960  12.118309   

                                      Low             ...       Open  \
                 MSFT      NVDA      AAPL      GOOGL  ...       MSFT   
Date                                                  ...              
2007-01-03  21.349220  0.573296  2.453970  11.444700  ...  21.109262   
2007-01-04  21.151600  0.551293  2.511499  11.624395  ...  20.961047   
2007-01-05  20.996335  0.537847  2.528879  11.866638  ...  20.911643   
2007-01-08  21.243346  0.528068  2.555246  11.968149  ...  20.925754   
2007-01-09  21.299809  0.522414  2.551350  11.943330  ...  21.172772   

                          Volume                                  Outlier  \
                NVDA        AAPL      GOOGL      MSFT        NVDA    AAPL   
Date                                                                        
2007-01-03  0.566420  1238319600  307951740  76935100  1154820000   False   
2007-01-04  0.549307   847260400  315188496  45774500   797298000   False   
2007-01-05  0.535708   834741600  274609116  44607200  1243344000   False   
2007-01-08  0.516150   797106800  189985824  50220200   657270000   False   
2007-01-09  0.518900  3349298400  215040744  44636600   764166000   False   

                                 
            GOOGL   MSFT   NVDA  
Date                             
2007-01-03  False  False  False  
2007-01-04  False  False  False  
2007-01-05  False  False  False  
2007-01-08  False  False  False  
2007-01-09  False  False  False  

[5 rows x 24 columns]